**Sample ID**: 1068




**Query**: Forward the titles and links of all Confluence pages about onboarding, created in the last 7 days, via Gmail to hr@brightmedia.in with the subject "Onboarding Resources".




**DB Type**: Base Case




**Case Description**: The Confluence database contains at least one page created within the last 7 days whose title includes the keyword “onboarding” (case-insensitive). All such pages have a valid link field. There may also be other pages created in the same period that are not related to onboarding.




**Global/Context Variables**:

- email_id = "hr@brightmedia.in"
- email_subject = "Onboarding Resources"
- current_date = "2025-04-23"
- time_delta_days = 7






**APIs**:

- confluence
- gmail


# Set Up

## Download relevant files

In [2]:
import io
import os
import sys
import zipfile
import shutil
import re
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Version to download
VERSION = "0.1.4"  # Version of the API

# Define paths
CONTENT_DIR = '/content'
APIS_DIR = os.path.join(CONTENT_DIR, 'APIs')
DBS_DIR = os.path.join(CONTENT_DIR, 'DBs')
SCRIPTS_DIR = os.path.join(CONTENT_DIR, 'Scripts')
FC_DIR = os.path.join(CONTENT_DIR, 'Schemas')
ZIP_PATH = os.path.join(CONTENT_DIR, f'APIs_V{VERSION}.zip')

# Google Drive Folder ID where versioned APIs zip files are stored
APIS_FOLDER_ID = '1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4'

# List of items to extract from the zip file
ITEMS_TO_EXTRACT = ['APIs/', 'DBs/', 'Scripts/', 'Schemas/']

# Clean up existing directories and files
for path in [APIS_DIR, DBS_DIR, SCRIPTS_DIR, FC_DIR, ZIP_PATH]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

# Authenticate and create the drive service
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Helper function to download a file from Google Drive
def download_drive_file(service, file_id, output_path, file_name=None, show_progress=True):
    """Downloads a file from Google Drive"""
    destination = output_path
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(destination, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if show_progress:
                print(f"Download progress: {int(status.progress() * 100)}%")


# 1. List files in the specified APIs folder
print(f"Searching for APIs zip file with version {VERSION} in folder: {APIS_FOLDER_ID}...")
apis_file_id = None

try:
    query = f"'{APIS_FOLDER_ID}' in parents and trashed=false"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get('files', [])
    for file in files:
        file_name = file.get('name', '')
        if file_name.lower() == f'apis_v{VERSION.lower()}.zip':
            apis_file_id = file.get('id')
            print(f"Found matching file: {file_name} (ID: {apis_file_id})")
            break

except Exception as e:
    print(f"An error occurred while listing files in Google Drive: {e}")

if not apis_file_id:
    print(f"Error: Could not find APIs zip file with version {VERSION} in the specified folder.")
    sys.exit("Required APIs zip file not found.")

# 2. Download the found APIs zip file
print(f"Downloading APIs zip file with ID: {apis_file_id}...")
download_drive_file(drive_service, apis_file_id, ZIP_PATH, file_name=f'APIs_V{VERSION}.zip')

# 3. Extract specific items from the zip file to /content
print(f"Extracting specific items from {ZIP_PATH} to {CONTENT_DIR}...")
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()

        for member in zip_contents:
            extracted = False
            for item_prefix in ITEMS_TO_EXTRACT:
              if member == item_prefix or member.startswith(item_prefix):
                    zip_ref.extract(member, CONTENT_DIR)
                    extracted = True
                    break

except zipfile.BadZipFile:
    print(f"Error: The downloaded file at {ZIP_PATH} is not a valid zip file.")
    sys.exit("Invalid zip file downloaded.")
except Exception as e:
    print(f"An error occurred during extraction: {e}")
    sys.exit("Extraction failed.")


# 4. Clean up
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# 5. Add APIs to path
if os.path.exists(APIS_DIR):
    sys.path.append(APIS_DIR)
else:
    print(f"Error: APIS directory not found at {APIS_DIR} after extraction. Cannot add to path.")

# 6. Quick verification
# Check for the presence of the extracted items
verification_paths = [APIS_DIR, DBS_DIR, SCRIPTS_DIR]
all_present = True
print("\nVerifying extracted items:")
for path in verification_paths:
    if os.path.exists(path):
        print(f"? {path} is present.")
    else:
        print(f"? {path} is MISSING!")
        all_present = False

if all_present:
    print(f"\n? Setup complete! Required items extracted to {CONTENT_DIR}.")
else:
    print("\n? Setup failed! Not all required items were extracted.")

# 7. Generate Schemas

print("\nGenerating FC Schemas")

# Change working directory to the source folder

# Iterate through the packages in the /content/APIs directory

    # Check if it's a directory (to avoid processing files)
        # Call the function to generate schema for the current package
print(f"? Successfully generated {len(os.listdir(FC_DIR))} FC Schemas to {FC_DIR}")
os.chdir(CONTENT_DIR)

Searching for APIs zip file with version 0.1.4 in folder: 1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4...
Found matching file: APIs_V0.1.4.zip (ID: 1TnAaWGfVrMxWTilyhy46-Aue_bh0XkNk)
Download progress: 100%
Extracting specific items from /content/APIs_V0.1.4.zip to /content...

Verifying extracted items:
? /content/APIs is present.
? /content/DBs is present.
? /content/Scripts is present.

? Setup complete! Required items extracted to /content.

Generating FC Schemas
? Successfully generated 70 FC Schemas to /content/Schemas


## Install Dependencies and Clone Repositories

In [ ]:
!pip install -r /content/APIs/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 83.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 97.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.6/343.6 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.1/245.1 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.9/443.9 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.6/245.6 kB 17.7 MB/s eta 

## Import APIs and initiate DBs

In [ ]:
import confluence
import gmail
import os
import random
import string

# Load API States
confluence.SimulationEngine.db.load_state("/content/DBs/ConfluenceDefaultDB.json")
gmail.SimulationEngine.db.load_state("/content/DBs/GmailDefaultDB.json")

# Global variables
email_id = "hr@brightmedia.in"
email_subject= "Onboarding Resources"

# Local variables
confluence_space_key="HR"

# Create the dummy Confluence space
dummy_space = confluence.create_space({
    "key": confluence_space_key,
    "name": "HR Resources Confluence Space",
    "description": "This is a HR space for onboarding and other resources."
})
print("Created dummy space:", confluence_space_key)

# Manually assigned posting dates
posting_day_1 = "2025-04-21"  # 2 days ago
posting_day_2 = "2025-04-19"  # 4 days ago
posting_day_3 = "2025-04-17"  # 6 days ago
posting_day_4 = "2025-04-18"  # 5 days ago
posting_day_5 = "2025-04-22"  # 1 days ago

# Page 1 - with "onboarding"
new_id_1 = ''.join(random.choices(string.ascii_lowercase + string.digits, k=8))
page_1 = {
    "id": new_id_1,
    "type": "page",
    "spaceKey": confluence_space_key,
    "title": "Onboarding Checklist",
    "status": "current",
    "body": {
        "storage": {
            "value": "<p>This is the content for the page <b>Onboarding Checklist</b>.</p>",
            "representation": "storage"
        }
    },
    "postingDay": posting_day_1,
    "link": os.path.join("/content", new_id_1)
}
confluence.create_content(body=page_1)
print("Created page:", page_1["title"])

# Page 2 - with "onboarding"
new_id_2 = ''.join(random.choices(string.ascii_lowercase + string.digits, k=8))
page_2 = {
    "id": new_id_2,
    "type": "page",
    "spaceKey": confluence_space_key,
    "title": "Employee Onboarding FAQ",
    "status": "current",
    "body": {
        "storage": {
            "value": "<p>This is the content for the page <b>Employee Onboarding FAQ</b>.</p>",
            "representation": "storage"
        }
    },
    "postingDay": posting_day_2,
    "link": os.path.join("/content", new_id_2)
}
confluence.create_content(body=page_2)
print("Created page:", page_2["title"])

# Page 3 - with "onboarding"
new_id_3 = ''.join(random.choices(string.ascii_lowercase + string.digits, k=8))
page_3 = {
    "id": new_id_3,
    "type": "page",
    "spaceKey": confluence_space_key,
    "title": "Onboarding Welcome Guide",
    "status": "current",
    "body": {
        "storage": {
            "value": "<p>This is the content for the page <b>Onboarding Welcome Guide</b>.</p>",
            "representation": "storage"
        }
    },
    "postingDay": posting_day_3,
    "link": os.path.join("/content", new_id_3)
}
confluence.create_content(body=page_3)
print("Created page:", page_3["title"])

# Page 4 - not onboarding
new_id_4 = ''.join(random.choices(string.ascii_lowercase + string.digits, k=8))
page_4 = {
    "id": new_id_4,
    "type": "page",
    "spaceKey": confluence_space_key,
    "title": "Holiday Policy",
    "status": "current",
    "body": {
        "storage": {
            "value": "<p>This is the content for the page <b>Holiday Policy</b>.</p>",
            "representation": "storage"
        }
    },
    "postingDay": posting_day_4,
    "link": os.path.join("/content", new_id_4)
}
confluence.create_content(body=page_4)
print("Created page:", page_4["title"])

# Page 5 - not onboarding
new_id_5 = ''.join(random.choices(string.ascii_lowercase + string.digits, k=8))
page_5 = {
    "id": new_id_5,
    "type": "page",
    "spaceKey": confluence_space_key,
    "title": "Performance Review Process",
    "status": "current",
    "body": {
        "storage": {
            "value": "<p>This is the content for the page <b>Performance Review Process</b>.</p>",
            "representation": "storage"
        }
    },
    "postingDay": posting_day_5,
    "link": os.path.join("/content", new_id_5)
}
confluence.create_content(body=page_5)
print("Created page:", page_5["title"])

# Initial Assertion
1. Assert that there is at least one Confluence page with "onboarding" (case-insensitive) in its title that was created in last 7 days with corresponding links.
2. Assert that no email has been sent to "hr@brightmedia.in" with the subject "Onboarding Resources" containing the above pages titles and links.

In [ ]:
from Scripts.assertions_utils import *
import confluence
import gmail
import re
from datetime import datetime, timedelta

# Define constants
email_id = "hr@brightmedia.in"
email_subject= "Onboarding Resources"
time_delta_days = 7
current_date = "2025-04-23"

# Locals
search_term = "onboarding"
# Compute date 7 days ago
cutoff_end = datetime.fromisoformat(current_date) - timedelta(days=time_delta_days)

# Get all Confluence pages from the simulated DB
search_results = confluence.get_content_list()
search_results = [element for element in search_results if compare_strings(element['type'], 'page')]

# Filter pages that match title, are recent, and contain a link
filtered_results = [
    p for p in search_results
    if re.search(search_term, p.get("title"), re.IGNORECASE)
]

filtered_results = [
    result for result in filtered_results
    if compare_datetimes(datetime.fromisoformat(result.get("postingDay")), cutoff_end, "gte")
]

# Assert that at least one valid page exists
actual_size = len(filtered_results)
assert actual_size > 0, f"Initial Assertion Failed: Expected at least one Confluence page with '{search_term}' in the title created in the last {time_delta_days} days, but found {actual_size}."

# Get all emails sent to the recipient with the subject
email_response = gmail.list_messages(q=f"subject:{email_subject} to:{email_id}")
email_messages = email_response.get("messages", [])

# Assert that none of these emails contain the titles and links of the qualifying Confluence pages.
matched_emails = [
    msg for msg in email_messages
    if any(
        compare_is_string_subset(p.get('title', ''), msg.get('body', ''))
        and compare_is_string_subset(p.get('link', ''), msg.get('body', ''))
        for p in filtered_results)
]
assert not matched_emails, f"Initial Assertion Failed: Email sent to {email_id} with subject '{email_subject}' contains the titles and links of the qualifying Confluence pages."

# Action
- Get the titles and links of all Confluence pages about onboarding, created in the last 7 days
- Send via Gmail to hr@brightmedia.in with the subject "Onboarding Resources".

In [ ]:
import confluence
import gmail
from datetime import datetime, timedelta

# Global variables
email_id = "hr@brightmedia.in"
email_subject= "Onboarding Resources"
time_delta_days = 7
current_date = "2025-04-23"

# Compute date 7 days ago
cutoff_end = datetime.fromisoformat(current_date) - timedelta(days=time_delta_days)

# Step 1: Fetch all created pages from the space
all_pages = confluence.get_content_list()
cutoff_pages = [
    page for page in all_pages
    if page.get('postingDay')
    and datetime.fromisoformat(page['postingDay']) >= cutoff_end
]
# Step 2: Print all pages
if not cutoff_pages:
    print("No pages found in the last 7 days.")
else:
    print("\nFiltered Pages:")
    for page in cutoff_pages:
        print(f"- {page['title']} | Posted: {page.get('postingDay', 'N/A')} | Link: {page.get('link', 'N/A')}")

In [ ]:
# Step 1: Hardcoded filtered onboarding pages created in the last 7 days
filtered_pages = [
    {
        "title": "Onboarding Checklist",
        "link": "/content/6"
    },
    {
        "title": "Employee Onboarding FAQ",
        "link": "/content/7"
    },
    {
        "title": "Onboarding Welcome Guide",
        "link": "/content/8"
    }
]

# Step 2: Handle condition logic
if filtered_pages:
    # Compose the email body
    email_body = "Hi HR Team,\n\nHere are the Confluence pages related to onboarding created in the last 7 days:\n\n"
    for page in filtered_pages:
        email_body += f"- {page['title']}: {page['link']}\n"
    email_body += "\nBest regards."

    # Compose the Gmail message
    email_message = {
        "recipient": email_id,
        "subject": email_subject,
        "body": email_body
    }

    # Fetch all previously sent messages
    sent_message_response = gmail.list_messages(q=f"subject:{email_subject} to:{email_id}")
    sent_messages = sent_message_response.get("messages", [])

    # Check for duplicate based on recipient and subject
    duplicate_found = any(
        all(page['title'].lower() in msg.get('body', '').lower() and page['link'] in msg.get('body', '') for page in filtered_pages)
        for msg in sent_messages
    )

    if duplicate_found:
        print(f"?? Email was already sent to {email_message['recipient']} with subject '{email_message['subject']}'.")
    else:
        # Attempt to send the email, handle Gmail permission errors
        response = gmail.send_message(msg=email_message)
        if response.get("id"):
            print(f"? Email sent to {email_message['recipient']} with subject '{email_message['subject']}'.")
        else:
            print(f"? Email not sent: Got {response}")
else:
    # No qualifying onboarding pages in last 7 days
    print("? No onboarding pages found in the last 7 days. Email will not be sent.")

# Final Assertion
1. Assert that an email has been sent from the user's Gmail account to hr@brightmedia.in with the subject "Onboarding Resources", and the email body includes the titles and links of all Confluence pages created in the last 7 days where the title contains the word "onboarding" (case-insensitive).

In [ ]:
from Scripts.assertions_utils import *
import gmail
import html
import confluence
from datetime import datetime, timedelta
import re

# === Constants ===
recipient = "hr@brightmedia.in"
subject = "Onboarding Resources"
search_term = "onboarding"
time_delta_days = 7
current_date = "2025-04-23"

# Compute date 7 days ago
cutoff_end = datetime.fromisoformat(current_date) - timedelta(days=time_delta_days)

# === Get all Confluence pages using the API ===
search_results = confluence.get_content_list(type="page")

# === Filter for onboarding pages created within the last 7 days ===
expected_lines = [
    (p['title'], p['link'])
    for p in search_results
    if compare_is_list_subset("postingDay", list(p.keys())) and compare_is_list_subset("title", list(p.keys())) and compare_is_list_subset("link", list(p.keys()))
    and compare_datetimes(datetime.fromisoformat(p["postingDay"]), cutoff_end, "gte")
    and re.search(search_term, p["title"], re.IGNORECASE)
]


# Filter only sent messages
messages = [
    m for m in gmail.list_messages().get("messages", [])
    if "SENT" in m.get("labelIds", [])
]

# Match email to recipient + subject
matched = []
for msg in messages:
    if compare_strings(msg.get("recipient"), recipient) and compare_strings(msg.get("subject"), subject):
        body_text = html.unescape(msg.get("body", "")).lower()

        # Require ALL titles and links in the body
        if all(
            compare_is_string_subset(title.lower(), body_text) and
            compare_is_string_subset(link.lower(), body_text)
            for title, link in expected_lines
        ):
            matched.append(msg)

assert matched, (
    f"Final Assertion Failed: No sent email to {recipient} with subject '{subject}' "
    "contained all expected onboarding page titles and links."
)
